In [ ]:
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18
from torchvision.models.resnet import BasicBlock
import torch.ao.quantization as quant
import types
import torch.ao.quantization as aq
from torchvision.models.resnet import BasicBlock
from torch.ao.quantization import get_default_qat_qconfig
from torch.ao.quantization.quantize_fx import prepare_qat_fx, convert_fx

import os
import logging
from datetime import datetime

In [ ]:
os.makedirs("./data", exist_ok=True)
os.makedirs("./logs", exist_ok=True)
os.makedirs("./trained_models", exist_ok=True)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), 
                         (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), 
                         (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=100,
                                         shuffle=False, num_workers=2)

## Full ResNet18 Traininig

In [ ]:
model = resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

In [ ]:
def training_loop(model, model_name, trainloader, testloader, num_epochs = 10):
    start_of_training_timestamp = datetime.now().strftime("%d.%m.%Y-%H:%M:%S")
    log_filename = f"./logs/{model_name}_{start_of_training_timestamp}.log"

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(message)s",
        handlers=[
            logging.FileHandler(log_filename),
            logging.StreamHandler()
        ]
    )

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.1)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        scheduler.step()
        
        train_loss = running_loss / total
        train_acc = 100. * correct / total
        
        model.eval()
        test_loss = 0.0
        correct_test = 0
        total_test = 0
        with torch.no_grad():
            for inputs, labels in testloader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                test_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                total_test += labels.size(0)
                correct_test += predicted.eq(labels).sum().item()
        
        test_loss /= total_test
        test_acc = 100. * correct_test / total_test
        
        # Log metrics
        logging.info(
            f"Epoch [{epoch+1}/{num_epochs}] "
            f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% "
            f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%"
        )
    return start_of_training_timestamp

In [ ]:
start_of_training_timestamp = training_loop(model, "resnet18_cifar", trainloader, testloader)

In [ ]:
model_path = f"./trained_models/resnet18_cifar10_{start_of_training_timestamp}.pth"
torch.save(model.state_dict(), model_path)
print(f"Model saved as {model_path}")

size_bytes = os.path.getsize(model_path)
size_mb = size_bytes / (1024 * 1024)
print(f"Model size: {size_mb:.2f} MB")

## QAT ResNet Training

In [ ]:
qat_model = resnet18()
qat_model.fc = nn.Linear(qat_model.fc.in_features, 10)

if isinstance(trainset, torchvision.datasets.CIFAR10):
    qat_model.conv1 = nn.Conv2d(
        in_channels=3,
        out_channels=64,
        kernel_size=3,
        stride=1,          
        padding=1,          
        bias=False
    )

    # Remove maxpool (not needed for small inputs)
    qat_model.maxpool = nn.Identity()

In [ ]:
print(qat_model)

In [ ]:
class QuantizableBasicBlock(nn.Module):
    def __init__(self, orig_block: BasicBlock):
        super().__init__()
        # reuse original parameters / submodules
        self.conv1 = orig_block.conv1
        self.bn1 = orig_block.bn1
        self.relu = orig_block.relu
        self.conv2 = orig_block.conv2
        self.bn2 = orig_block.bn2
        self.downsample = orig_block.downsample  # may be None
        self.stride = orig_block.stride

        # local quant/dequant stubs (these use observers when prepared)
        self.quant = aq.QuantStub()
        self.dequant = aq.DeQuantStub()

    def forward(self, x):
        identity = x

        # Quantize at block entry => convs will be fake-quantized during QAT
        out = self.quant(x)

        out = self.conv1(out)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        # Dequantize so addition runs in FP32
        out = self.dequant(out)

        if self.downsample is not None:
            # keep downsample in FP32 by applying it directly on FP32 input
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)
        return out

In [ ]:
def make_blocks_quantizable(model):
    for layer_name in ['layer1', 'layer2', 'layer3', 'layer4']:
        layer = getattr(model, layer_name)
        for i in range(len(layer)):
            orig_block = layer[i]
            layer[i] = QuantizableBasicBlock(orig_block)

make_blocks_quantizable(qat_model)

In [ ]:
# # https://docs.pytorch.org/docs/stable/generated/torch.ao.quantization.fuse_modules.fuse_modules.html
# '''
# The fused module implements the combined operation more efficiently 
# and with better numerical stability under quantization.
# Still under question if I really need it.
# '''

# qat_model.eval()

# try:
#     # this may succeed or warn depending on module naming; wrap in try
#     aq.fuse_modules(qat_model, [['conv1', 'bn1', 'relu']], inplace=True)
# except Exception as e:
#     print("Top-level fuse warning:", e)

# # Fuse within each wrapped block
# for layer_name in ['layer1', 'layer2', 'layer3', 'layer4']:
#     layer = getattr(qat_model, layer_name)
#     for i in range(len(layer)):
#         block = layer[i]
#         # block has attributes conv1, bn1, relu, conv2, bn2
#         try:
#             aq.fuse_modules(block, [['conv1', 'bn1', 'relu'], ['conv2', 'bn2']], inplace=True)
#         except Exception as e:
#             print(f"Fuse warning for {layer_name}.{i}:", e)

In [ ]:
default_qconfig = get_default_qat_qconfig('fbgemm')
qat_model.qconfig = default_qconfig

qat_model.conv1.qconfig = None
qat_model.fc.qconfig = None

for layer_name in ['layer1', 'layer2', 'layer3', 'layer4']:
    layer = getattr(qat_model, layer_name)
    for block in layer:
        if getattr(block, 'downsample', None) is not None:
            # set downsample (Sequential) to FP32
            block.downsample.qconfig = None

In [ ]:
example_inputs = torch.randn(1, 3, 32, 32)

qat_model.train()
qconfig_dict = {
    "": default_qconfig,                # default for most modules
    "conv1": None,                      # keep conv1 FP32
    "fc": None,                         # keep fc FP32
    "downsample": None,
}
qat_model = prepare_qat_fx(qat_model, qconfig_dict, example_inputs)
qat_model.to(device)

In [ ]:
start_of_training_timestamp = training_loop(qat_model, "resnet18_cifar10_qat", trainloader, testloader, num_epochs=5)

In [ ]:
start_of_training_timestamp = "2025-10-07_11:53:43"
qat_model.eval()
qat_model.to("cpu")
quantized_model = convert_fx(qat_model)

qt_path = f"./trained_models/resnet18_cifar10_qat_{start_of_training_timestamp}.pth"
torch.save(quantized_model.state_dict(), qt_path)

print(f"Quantized model saved as {qt_path}")
print("Quantized file size (MB):", os.path.getsize(qt_path)/(1024**2))

In [ ]:
print(quantized_model)

In [ ]:
qat_model.eval()
test_loss = 0.0
correct_test = 0
total_test = 0
with torch.no_grad():
    for inputs, labels in testloader:
        inputs, labels = inputs.to('cpu'), labels.to('cpu')
        outputs = qat_model(inputs)
        criterion = nn.CrossEntropyLoss()
        loss = criterion(outputs, labels)
        test_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total_test += labels.size(0)
        correct_test += predicted.eq(labels).sum().item()

test_loss /= total_test
test_acc = 100. * correct_test / total_test

# Log metrics
logging.info(
    f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%"
)